# 06 — SAE Feature Interpretation & Visualization

Interpret the Sparse Autoencoder trained in `06_training_sae.ipynb`.  
**Input**: SAE checkpoint + activation H5 file (from `06_collect_activations.ipynb`) + fine-tuned ViT checkpoint.  
**Output**: feature statistics, top-K patch visualizations, spatial heatmaps, UMAP, ablation analysis.

## 0. Kaggle Setup

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru h5py umap-learn
!pip install -q PyDrive2 scikit-learn

## Google Drive

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## Paths

In [ ]:
import os
from pathlib import Path

# ← update these
H5_PATH      = '/kaggle/input/activations/deit_base_patch16_layer9_acts.h5'
SAE_CKPT     = '/kaggle/input/sae/sae_deit_base_patch16_layer9_best.pt'
VIT_CKPT     = '/kaggle/input/models/youssefnouiouar1/crt-deit/pytorch/default/1/deit_base_patch16_best.pth'
TRAINVAL_ROOT = Path('/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K')
TEST_ROOT     = Path('/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K')
PROJECT_ROOT  = '/kaggle/working/xai-vit-medical'

SAVE_DIR = '/kaggle/working/interpretation'
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. Imports & Config

In [ ]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import h5py
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from tqdm.notebook import tqdm
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import umap
import timm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed

SEED        = 42
set_seed(SEED)
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── SAE / model params ────────────────────────────────────────────────────────
D_INPUT      = 768
D_SAE        = 768 * 16    # 12 288
K            = 32
LAYER_IDX    = 9
NUM_CLASSES  = 9
CLASS_NAMES  = list(DEFAULT_CRC_CLASSES)
PATCH_GRID   = 14           # 14×14 grid (DeiT/ViT-Base with 16px patches)
PATCH_SIZE   = 16           # pixels per patch
IMAGE_SIZE   = 224

# visualization subset
SAMPLES_PER_CLASS = 50      # images per class for visualization
TOP_FEATS_PER_CLASS = 3     # how many top features to inspect per class

print(f'Device : {DEVICE}')

## 2. Load SAE

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_input, d_sae, k):
        super().__init__()
        self.d_input = d_input
        self.d_sae   = d_sae
        self.k       = k
        self.W_enc = nn.Parameter(torch.empty(d_input, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_sae,  d_input))
        self.b_dec = nn.Parameter(torch.zeros(d_input))

    def encode(self, x):
        pre  = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre)
        acts.scatter_(-1, idx, vals.clamp(min=0))
        return acts

    def decode(self, acts):
        return acts @ self.W_dec + self.b_dec

    def forward(self, x):
        acts  = self.encode(x)
        return self.decode(acts), acts


ckpt_sae = torch.load(SAE_CKPT, map_location='cpu')
cfg      = ckpt_sae['config']

sae = TopKSAE(cfg['d_input'], cfg['d_sae'], cfg['k'])
sae.W_enc.data = ckpt_sae['W_enc']
sae.b_enc.data = ckpt_sae['b_enc']
sae.W_dec.data = ckpt_sae['W_dec']
sae.b_dec.data = ckpt_sae['b_dec']
sae = sae.to(DEVICE).eval()

MEAN = ckpt_sae['mean'].to(DEVICE)   # (768,) normalization mean
STD  = ckpt_sae['std'].to(DEVICE)    # (768,) normalization std

print(f'SAE loaded  : d_input={cfg["d_input"]}  d_sae={cfg["d_sae"]}  k={cfg["k"]}')
print(f'Best epoch  : step {ckpt_sae["step"]}')

## 3. Streaming Metrics over H5

Single pass over all activations to compute **reconstruction quality** (R², cosine similarity) and **feature activity** (activation frequency, class specificity).  
Nothing is fully loaded into RAM — processed in batches of 256 images.

In [ ]:
BATCH_IMG = 256

feat_fires       = torch.zeros(D_SAE)
class_fires      = torch.zeros(NUM_CLASSES, D_SAE)
class_count_tens = torch.zeros(NUM_CLASSES)
ss_res = ss_tot = cos_sum = n_tokens = 0.0

with h5py.File(H5_PATH, 'r') as f:
    N_img, P, D = f['activations'].shape
    labels_h5   = f['labels'][:]

    for i in tqdm(range(0, N_img, BATCH_IMG), desc='Metrics pass'):
        acts_np  = f['activations'][i:i+BATCH_IMG].astype(np.float32)  # (B, 196, 768)
        labels_b = labels_h5[i:i+BATCH_IMG]
        B        = len(acts_np)

        tokens = torch.from_numpy(acts_np.reshape(B*P, D)).to(DEVICE)
        x      = (tokens - MEAN) / STD

        with torch.no_grad():
            codes = sae.encode(x)      # (B*P, D_SAE)
            x_hat = sae.decode(codes)  # (B*P, 768)

        fires = (codes > 0).float()    # (B*P, D_SAE) on GPU
        feat_fires += fires.sum(dim=0).cpu()

        # class-level counts via one-hot scatter
        labels_rep   = torch.from_numpy(np.repeat(labels_b, P)).long()
        onehot       = F.one_hot(labels_rep, NUM_CLASSES).float()   # (B*P, 9)
        class_fires += (onehot.T @ fires.cpu())                     # (9, D_SAE)
        class_count_tens += onehot.sum(dim=0)

        ss_res  += ((x - x_hat)**2).sum().item()
        ss_tot  += ((x - x.mean(0, keepdim=True))**2).sum().item()
        cos_sum += F.cosine_similarity(x, x_hat).sum().item()
        n_tokens += B * P

        del tokens, x, codes, x_hat, fires

feat_freq  = (feat_fires / n_tokens).numpy()                              # (D_SAE,)
class_freq = (class_fires / class_count_tens.unsqueeze(1)).numpy()        # (9, D_SAE)
r2         = float(1 - ss_res / (ss_tot + 1e-8))
cos_sim    = float(cos_sum / n_tokens)
dead_pct   = float((feat_freq < 1e-4).mean() * 100)

print(f'R²              : {r2:.4f}   (target > 0.85)')
print(f'Cosine sim      : {cos_sim:.4f}   (target > 0.90)')
print(f'Dead features   : {dead_pct:.1f}%   (healthy: 10-30%)')
print(f'Tokens processed: {int(n_tokens):,}')

## 4. Class Selectivity & Top Features per Class

In [ ]:
# class_freq: (9, D_SAE) — activation freq of each feature per class
best_class = class_freq.argmax(axis=0)      # (D_SAE,) — which class activates each feature most
max_freq   = class_freq.max(axis=0)         # (D_SAE,)

# mean activation freq across the OTHER classes
mean_other = (class_freq.sum(axis=0) - max_freq) / (NUM_CLASSES - 1)

selectivity = (max_freq - mean_other) / (max_freq + mean_other + 1e-8)  # (D_SAE,)

# Top features per class
# BUG FIX: rank within class-c features only.
# The old approach (selectivity * class_mask) gives score=0 to non-class-c features.
# When all class-c features have negative selectivity they rank BELOW the zeros,
# so argsort picked random non-class-c features instead.
top_features_per_class = {}
features_to_viz = set()

print(f'{"Class":6s}  {"Top feat":>10s}  {"selectivity":>12s}  {"freq":>8s}')
print('-' * 44)
for c, cname in enumerate(CLASS_NAMES):
    # Only consider features whose best class is c
    class_feat_idx = np.where(best_class == c)[0]   # (n_c,) feature indices

    if len(class_feat_idx) > 0:
        order   = np.argsort(selectivity[class_feat_idx])[::-1][:TOP_FEATS_PER_CLASS]
        top_idx = class_feat_idx[order]
    else:
        # Fallback: features that fire most often for this class (no argmax winner)
        top_idx = np.argsort(class_freq[c])[::-1][:TOP_FEATS_PER_CLASS]

    top_features_per_class[cname] = top_idx
    features_to_viz.update(top_idx.tolist())
    print(f'{cname:6s}  feat #{top_idx[0]:>6d}   sel={selectivity[top_idx[0]]:.3f}   freq={max_freq[top_idx[0]]:.4f}')

features_to_viz = sorted(features_to_viz)
print(f'\nTotal features to visualize: {len(features_to_viz)}')

## 5. Feature Statistics Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Activation frequency
freq_pct = feat_freq * 100
axes[0].hist(freq_pct[freq_pct > 0], bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(0.01, color='red',    ls='--', label='0.01% near-dead')
axes[0].axvline(5.0,  color='orange', ls='--', label='5% universal')
axes[0].set(xlabel='Activation frequency (%)', ylabel='# Features',
            title='Feature Activation Frequency', xscale='log')
axes[0].legend(fontsize=8)

# Class selectivity
axes[1].hist(selectivity[selectivity > 0], bins=60, color='seagreen', edgecolor='none')
axes[1].axvline(0.5, color='red', ls='--', label='sel > 0.5')
axes[1].set(xlabel='Selectivity index', ylabel='# Features', title='Class Selectivity')
axes[1].legend(fontsize=8)

# Top-1 feature per class — show selectivity and annotate with feature index
top1_sel  = [selectivity[top_features_per_class[c][0]] for c in CLASS_NAMES]
top1_feat = [int(top_features_per_class[c][0])         for c in CLASS_NAMES]
bars = axes[2].barh(CLASS_NAMES, top1_sel, color='tomato')
for bar, feat_i, sel_val in zip(bars, top1_feat, top1_sel):
    x_label = sel_val + 0.01 if sel_val >= 0 else 0.01
    axes[2].text(x_label, bar.get_y() + bar.get_height() / 2,
                 f'feat #{feat_i}', va='center', fontsize=7)
axes[2].set(xlabel='Selectivity (top-1 feature)', title='Top Feature per Class')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'feature_stats.png'), dpi=150)
plt.show()

## 6. Downstream Task Recovery

Train a logistic regression on SAE-reconstructed activations vs. original activations.  
Measures how much diagnostic information the SAE preserves.

In [ ]:
X_orig  = []
X_recon = []
y_probe = []

with h5py.File(H5_PATH, 'r') as f:
    N_img, P, D = f['activations'].shape
    labels_h5   = f['labels'][:]

    for i in tqdm(range(0, N_img, BATCH_IMG), desc='Probe features'):
        acts_np  = f['activations'][i:i+BATCH_IMG].astype(np.float32)
        B        = len(acts_np)

        tokens = torch.from_numpy(acts_np.reshape(B*P, D)).to(DEVICE)
        x      = (tokens - MEAN) / STD

        with torch.no_grad():
            codes = sae.encode(x)
            x_hat = sae.decode(codes)

        # mean-pool over patches → image-level feature
        X_orig.append(x.reshape(B, P, D).mean(1).cpu().numpy())       # (B, 768)
        X_recon.append(x_hat.reshape(B, P, D).mean(1).cpu().numpy())   # (B, 768)
        y_probe.extend(labels_h5[i:i+BATCH_IMG].tolist())

        del tokens, x, codes, x_hat

X_orig  = np.vstack(X_orig)
X_recon = np.vstack(X_recon)
y_probe = np.array(y_probe)

# train on 80%, test on 20%
from sklearn.model_selection import train_test_split
idx_tr, idx_te = train_test_split(np.arange(len(y_probe)), test_size=0.2,
                                   random_state=SEED, stratify=y_probe)

clf_orig  = LogisticRegression(max_iter=500, C=1.0, n_jobs=-1)
clf_recon = LogisticRegression(max_iter=500, C=1.0, n_jobs=-1)
clf_orig.fit(X_orig[idx_tr],  y_probe[idx_tr])
clf_recon.fit(X_recon[idx_tr], y_probe[idx_tr])

acc_orig  = accuracy_score(y_probe[idx_te], clf_orig.predict(X_orig[idx_te]))
acc_recon = accuracy_score(y_probe[idx_te], clf_recon.predict(X_recon[idx_te]))

print(f'Linear probe — original activations : {acc_orig:.4f}')
print(f'Linear probe — SAE reconstructions  : {acc_recon:.4f}')
print(f'Information gap                      : {(acc_orig - acc_recon)*100:.1f}pp  (target < 3%)')

## 7. Visualization — Load ViT + Sample Images

We sample `SAMPLES_PER_CLASS` images per class from the val split, run them through the fine-tuned ViT + SAE,  
and store activations for the features identified in Section 4.

In [ ]:
# Load fine-tuned ViT
model = timm.create_model('deit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
ckpt_vit = torch.load(VIT_CKPT, map_location='cpu')
model.load_state_dict(ckpt_vit['model_state_dict'])
model = model.to(DEVICE).eval()
print('ViT loaded.')

# Hook on layer LAYER_IDX to capture patch-token activations
act_buffer = []
def hook_fn(module, inp, out):
    act_buffer.append(out[:, 1:, :].cpu())   # skip CLS token → (B, 196, 768)

hook = model.blocks[LAYER_IDX].register_forward_hook(hook_fn)

# Val dataset — same params as collection notebook
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.5,
    random_state=SEED,
)
val_ds = CRCHistologyDataset(split='val', splits=crc_splits,
                              image_size=IMAGE_SIZE, return_id=True)

# Sample SAMPLES_PER_CLASS per class
class_idx_map = defaultdict(list)
for i, lbl in enumerate(val_ds.labels):
    class_idx_map[int(lbl)].append(i)

rng_vis = np.random.default_rng(SEED + 1)
vis_indices = []
for c in range(NUM_CLASSES):
    chosen = rng_vis.choice(class_idx_map[c], SAMPLES_PER_CLASS, replace=False)
    vis_indices.extend(chosen.tolist())

vis_loader = DataLoader(Subset(val_ds, vis_indices),
                        batch_size=32, shuffle=False, num_workers=2)
print(f'Vis set: {len(vis_indices)} images  ({SAMPLES_PER_CLASS} per class)')

In [ ]:
feat_idx_t = torch.tensor(features_to_viz, dtype=torch.long)  # indices to store
n_viz      = len(features_to_viz)

vis_images_np   = []   # (N_vis, 224, 224, 3) uint8
vis_labels_list = []
vis_feat_acts   = []   # (N_vis, 196, n_viz)  — only top features
vis_mean_codes  = []   # (N_vis, D_SAE)       — for UMAP
vis_recon_mean  = []   # (N_vis, 768)         — for ablation

_imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
_imagenet_std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

with torch.no_grad():
    for imgs, lbls, _ in tqdm(vis_loader, desc='Forward pass'):
        imgs = imgs.to(DEVICE)
        B    = len(imgs)

        act_buffer.clear()
        model(imgs)
        layer_acts = act_buffer[0].to(DEVICE)   # (B, 196, 768)

        x     = (layer_acts.reshape(B*196, 768) - MEAN) / STD
        codes = sae.encode(x)                    # (B*196, D_SAE) on GPU
        x_hat = sae.decode(codes)                # (B*196, 768)

        codes3d = codes.view(B, 196, D_SAE)

        vis_feat_acts.append(codes3d[:, :, feat_idx_t].cpu().numpy())  # (B, 196, n_viz)
        vis_mean_codes.append(codes3d.mean(dim=1).cpu().numpy())       # (B, D_SAE)
        vis_recon_mean.append(x_hat.view(B, 196, 768).mean(1).cpu().numpy())  # (B, 768)

        imgs_disp = (imgs.cpu() * _imagenet_std + _imagenet_mean).clamp(0, 1)
        vis_images_np.extend((imgs_disp.permute(0,2,3,1).numpy()*255).astype(np.uint8))
        vis_labels_list.extend(lbls.tolist())

        del codes, x_hat, codes3d, x

hook.remove()

vis_feat_acts  = np.concatenate(vis_feat_acts,  axis=0)   # (N_vis, 196, n_viz)
vis_mean_codes = np.concatenate(vis_mean_codes, axis=0)   # (N_vis, D_SAE)
vis_recon_mean = np.concatenate(vis_recon_mean, axis=0)   # (N_vis, 768)
vis_labels_arr = np.array(vis_labels_list)
print(f'Collected: {vis_feat_acts.shape}  |  mean_codes: {vis_mean_codes.shape}')

## 8. Top-K Activating Patches

For each of the top class-specific features, show the 9 patches that activate it most strongly.  
If patches for one feature are visually coherent (all glandular lumens, all lymphocytes…) → **monosemantic**.

In [ ]:
def show_top_patches(feature_idx, k=9, title=None):
    """Display top-k patches that maximally activate a given feature."""
    if feature_idx not in features_to_viz:
        print(f'Feature {feature_idx} not in visualization set. Add it to features_to_viz.')
        return

    col      = features_to_viz.index(feature_idx)
    flat     = vis_feat_acts[:, :, col].reshape(-1)   # (N_vis * 196,)
    top_flat = np.argsort(flat)[::-1][:k]

    nrows = (k + 2) // 3
    fig, axes = plt.subplots(nrows, 3, figsize=(7, nrows * 2.5))
    axes = axes.flatten()

    for rank, flat_i in enumerate(top_flat):
        img_i   = flat_i // 196
        patch_i = flat_i %  196
        row_p   = patch_i // PATCH_GRID
        col_p   = patch_i %  PATCH_GRID
        y1, x1  = row_p * PATCH_SIZE, col_p * PATCH_SIZE
        patch   = vis_images_np[img_i][y1:y1+PATCH_SIZE, x1:x1+PATCH_SIZE]

        axes[rank].imshow(patch)
        axes[rank].set_title(f'act={flat[flat_i]:.2f}\n{CLASS_NAMES[vis_labels_arr[img_i]]}',
                              fontsize=7)
        axes[rank].axis('off')

    for ax in axes[k:]:
        ax.axis('off')

    suptitle = title or f'Feature #{feature_idx} — best class: {CLASS_NAMES[best_class[feature_idx]]}  sel={selectivity[feature_idx]:.3f}'
    plt.suptitle(suptitle, fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'topk_feat{feature_idx}.png'), dpi=150)
    plt.show()


# Show top-1 feature for each class
for cname in CLASS_NAMES:
    feat = top_features_per_class[cname][0]
    show_top_patches(feat, k=9, title=f'[{cname}] Feature #{feat}')

## 9. Spatial Heatmaps

For a given image and feature, reshape the 196 patch activations into a 14×14 grid  
and overlay as a heatmap on the original H&E image.

In [ ]:
def spatial_heatmap(img_np, acts_196, ax, alpha=0.45):
    """Overlay feature activation heatmap on image."""
    hm     = acts_196.reshape(PATCH_GRID, PATCH_GRID)
    hm_up  = np.kron(hm, np.ones((PATCH_SIZE, PATCH_SIZE)))    # 224×224
    hm_norm = (hm_up - hm_up.min()) / (hm_up.max() - hm_up.min() + 1e-8)
    ax.imshow(img_np)
    ax.imshow(cm.hot(hm_norm), alpha=alpha)
    ax.axis('off')


# For each class: show 3 sample images with the top-1 class feature heatmap
N_EXAMPLES = 3

for cname in CLASS_NAMES:
    feat    = top_features_per_class[cname][0]
    col     = features_to_viz.index(feat)
    c_idx   = CLASS_NAMES.index(cname)
    c_imgs  = np.where(vis_labels_arr == c_idx)[0][:N_EXAMPLES]

    fig, axes = plt.subplots(1, N_EXAMPLES, figsize=(N_EXAMPLES * 4, 4))
    for ax, img_i in zip(axes, c_imgs):
        spatial_heatmap(vis_images_np[img_i],
                        vis_feat_acts[img_i, :, col],
                        ax)
    plt.suptitle(f'[{cname}] Feature #{feat} — sel={selectivity[feat]:.3f}', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'heatmap_{cname}_feat{feat}.png'), dpi=150)
    plt.show()

## 10. UMAP of SAE Feature Space

Each image is represented by its mean SAE code (12 288-dim).  
PCA → 50 dims first, then UMAP → 2D.  
Color by tissue class: if TUM clusters separately from NORM, the SAE captures diagnostically relevant variation.

In [ ]:
X = vis_mean_codes   # (N_vis, D_SAE)
N_vis = len(X)

# Both PCA and UMAP have hard constraints relative to n_samples.
# Clamp to safe values so the cell works for any SAMPLES_PER_CLASS.
n_pca        = min(50, N_vis - 1)          # PCA: n_components must be < n_samples
n_neighbors  = min(15, max(5, N_vis // 6)) # UMAP: n_neighbors must be << n_samples

print(f'N_vis={N_vis}  →  n_pca={n_pca}  n_neighbors={n_neighbors}')

X_pca = PCA(n_components=n_pca, random_state=SEED).fit_transform(X)
X_2d  = umap.UMAP(n_components=2, random_state=SEED,
                   n_neighbors=n_neighbors, min_dist=0.1,
                   n_jobs=1).fit_transform(X_pca)

colors = plt.get_cmap('tab10')(np.linspace(0, 1, NUM_CLASSES))

fig, ax = plt.subplots(figsize=(10, 8))
for c, cname in enumerate(CLASS_NAMES):
    mask = vis_labels_arr == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               label=cname, color=colors[c], alpha=0.7, s=25, edgecolors='none')
ax.legend(markerscale=2, fontsize=9)
ax.set(title='UMAP — SAE Feature Space', xlabel='UMAP-1', ylabel='UMAP-2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'umap_sae_features.png'), dpi=150)
plt.show()

## 11. Ablation Test

Zero out the top class-specific feature for one class and measure how prediction changes.  
A linear probe is trained on the vis set reconstructions, then applied to ablated reconstructions.

In [ ]:
CLASS_TO_ABLATE = 'TUM'   # ← change to inspect other classes
feat_ablate     = int(top_features_per_class[CLASS_TO_ABLATE][0])
c_ablate        = CLASS_NAMES.index(CLASS_TO_ABLATE)

# Train a quick linear probe on vis_recon_mean
clf_vis = LogisticRegression(max_iter=500, C=1.0, n_jobs=-1)
clf_vis.fit(vis_recon_mean, vis_labels_arr)
acc_before = accuracy_score(vis_labels_arr, clf_vis.predict(vis_recon_mean))
print(f'Probe accuracy (before ablation): {acc_before:.4f}')

# Ablate: zero out feature j in mean codes, then decode
mean_codes_t = torch.from_numpy(vis_mean_codes).to(DEVICE)
codes_ablated = mean_codes_t.clone()
codes_ablated[:, feat_ablate] = 0.0

with torch.no_grad():
    x_recon_ablated = sae.decode(codes_ablated).cpu().numpy()   # (N_vis, 768)

preds_normal  = clf_vis.predict(vis_recon_mean)
preds_ablated = clf_vis.predict(x_recon_ablated)

# Analyse change for CLASS_TO_ABLATE images only
mask_c = vis_labels_arr == c_ablate
print(f'\nAblation of feature #{feat_ablate}  (top-{CLASS_TO_ABLATE} feature):')
print(f'  {CLASS_TO_ABLATE} images ({mask_c.sum()}):  before → after')

from collections import Counter
before_dist = Counter(CLASS_NAMES[p] for p in preds_normal[mask_c])
after_dist  = Counter(CLASS_NAMES[p] for p in preds_ablated[mask_c])
print(f'  Before: {dict(before_dist)}')
print(f'  After : {dict(after_dist)}')

# Fraction that flipped away from CLASS_TO_ABLATE
flipped = (preds_normal[mask_c] == c_ablate) & (preds_ablated[mask_c] != c_ablate)
print(f'  Flipped away from {CLASS_TO_ABLATE}: {flipped.sum()} / {mask_c.sum()}  ({flipped.mean()*100:.1f}%)')

## 12. Upload Results to Google Drive

In [ ]:
import glob

for fpath in glob.glob(os.path.join(SAVE_DIR, '*.png')):
    gd = drive.CreateFile({'title': os.path.basename(fpath)})
    gd.SetContentFile(fpath)
    gd.Upload()
    print(f'Uploaded: {os.path.basename(fpath)}')

print('Done.')